# Managed Agents: custom tools and structured errors

> **CCA-F Domain 2 - Tools and MCP (18%).** The exam tests that a tool failure returns a **structured error the model can read and retry**, not a raised exception that crashes the loop. A well-shaped `is_error` result lets the agent **retry once** instead of hallucinating an answer.

Theme: a **vintage guitar-gear price agent**. We define one custom tool, `gear_lookup`, that hits a flaky price source. The first call returns a **structured transient error**; the second call succeeds. The agent retries once and quotes a real number instead of inventing a Les Paul price.

### 1. Dependencies, environment, and client

Install deps, load `.env`, and create the client. The **Managed Agents API** lives under `client.beta`; every call carries the beta header via `betas=[BETA]`. **Haiku 4.5** is the repo default and handles custom tool use plus structured-error retry at production quality.

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import os, subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-haiku-4-5"          # repo default; Sonnet only where reasoning depth earns it
BETA = "managed-agents-2026-04-01"  # Managed Agents API beta gate

# Teardown guard. Default OFF so "Run All" in class leaves the agent and session
# live for Console inspection. Headless smoke tests set ARCHIVE_ON_RUN=1 to force
# cleanup so no billable resource dangles in CI. See the final cell.
RUN_TEARDOWN = os.environ.get("ARCHIVE_ON_RUN", "0") == "1"
print("Client ready.")

Client ready.


### 2. Define one custom tool and a flaky price source

We declare a single **custom tool**, `gear_lookup`, on the agent. A tool is a name, a description, and a JSON `input_schema`. The description is the model's only guidance on when to call the tool, so it earns real prose.

The lookup itself is deliberately **flaky**: the first call returns a **structured transient error**, the second returns a price. This is the retry drill in miniature.

> **Exam trap.** A tool that fails must return a **structured result the model can read** (an error marker in the content plus `is_error: true` on the event), never a raised Python exception. An exception crashes the loop; a structured error lets the agent decide to **retry once**.

In [2]:
import json

# The custom-tool declaration the agent carries. `input_schema` is standard JSON Schema.
# `type: "custom"` marks it a client-executed tool: when the agent calls it, the API emits
# an `agent.custom_tool_use` event and the session goes idle waiting for our result.
GEAR_LOOKUP_TOOL = {
    "type": "custom",
    "name": "gear_lookup",
    "description": (
        "Look up the current used-market price for a piece of vintage guitar gear "
        "by model name. Returns a price in USD. Call this before quoting any price; "
        "do not estimate from memory."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "model_name": {
                "type": "string",
                "description": "The gear model, e.g. '1959 Gibson Les Paul Standard'.",
            }
        },
        "required": ["model_name"],
    },
}

# A tiny in-notebook price book stands in for a real pricing service.
PRICE_BOOK = {"1959 gibson les paul standard": 275000}

# Call counter drives the flaky-then-healthy behavior without any real network.
_lookup_calls = {"n": 0}


def run_gear_lookup(model_name: str) -> dict:
    """Resolve a gear price, failing transiently on the first call.

    Returns a two-part signal so the caller can build the tool-result event:
      ok=False  -> structured transient error; set is_error=True on the event.
      ok=True   -> price payload; set is_error=False on the event.
    Why a dict, not a raise: the agent must read the failure and retry, not crash.
    """
    _lookup_calls["n"] += 1
    if _lookup_calls["n"] == 1:
        # First attempt: the source is briefly unavailable. Structured, retryable.
        return {
            "ok": False,
            "content": json.dumps(
                {
                    "error": "price source temporarily unavailable",
                    "errorCategory": "transient",
                    "isRetryable": True,
                }
            ),
        }
    price = PRICE_BOOK.get(model_name.strip().lower())
    return {"ok": True, "content": json.dumps({"model_name": model_name, "price_usd": price})}


print("Custom tool declared:", GEAR_LOOKUP_TOOL["name"])

Custom tool declared: gear_lookup


### 3. Stand up the agent, environment, and session

The Managed Agents spine is **agent -> environment -> session**. The agent carries the `tools` list and a `system` prompt that forbids guessing. The **session** is the runtime where the turn plays out; it holds memory server-side so we never resend prior turns.

In [3]:
agent = client.beta.agents.create(
    model={"id": MODEL},
    name="gear-scout",
    system=(
        "You are a vintage guitar-gear price desk. Never quote a price from memory. "
        "Always call gear_lookup first. If the tool returns a retryable error, call it "
        "one more time before answering."
    ),
    tools=[GEAR_LOOKUP_TOOL],
    betas=[BETA],
)

env = client.beta.environments.create(name="gear-scratch", betas=[BETA])

session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=env.id,
    betas=[BETA],
)

print("agent:", agent.id)
print("session:", session.id)

agent: agent_01XykSuqhFL3NkD25oJR9yqU
session: sesn_0182ZQY7sVv5qgSvrAssToCM


### 4. Run the turn: custom tool use, structured error, retry-once

Send the user turn, then **stream events until the session goes idle**. When the agent wants the tool it emits an **`agent.custom_tool_use`** event. We stop on **`session.status_idle`**; its `stop_reason.event_ids` names the tool-use event we must answer.

We reply with a **`user.custom_tool_result`** event, then stream to idle again. The first result carries `is_error=True`, so the agent calls the tool a second time and gets the price.

> **The distinction that trips people up.** The custom-tool result field is **`custom_tool_use_id`**, NOT `tool_use_id`. Built-in and MCP tools answer with `user.tool_result` + `tool_use_id`; **custom** tools answer with `user.custom_tool_result` + `custom_tool_use_id`. Mix them up and the event does not bind to the pending call.

In [4]:
def stream_to_idle(session_id):
    """Stream events until the session is idle. Return (idle_event, custom_tool_use_events).

    We collect any agent.custom_tool_use events so we can answer them, and we stop on
    session.status_idle - the exam-correct stop signal, not a token count or vibes.
    """
    custom_uses = []
    for event in client.beta.sessions.events.stream(session_id, betas=[BETA]):
        if event.type == "agent.custom_tool_use":
            custom_uses.append(event)
        elif event.type == "agent.message":
            print("agent:", event)
        elif event.type == "session.status_idle":
            return event, custom_uses
    return None, custom_uses


# 1. Send the user turn.
client.beta.sessions.events.send(
    session.id,
    events=[{
        "type": "user.message",
        "content": [{"type": "text", "text": "What is a 1959 Gibson Les Paul Standard worth?"}],
    }],
    betas=[BETA],
)

# 2. Drive the loop. The agent calls gear_lookup, hits the transient error, and retries once.
attempts = 0
while True:
    idle, custom_uses = stream_to_idle(session.id)
    if not custom_uses:
        break  # no pending tool call -> the agent answered in text

    # The idle event names which tool-use events await a result.
    pending_ids = set(getattr(idle.stop_reason, "event_ids", []) or [])

    results = []
    for use in custom_uses:
        if use.id not in pending_ids:
            continue
        attempts += 1
        outcome = run_gear_lookup(use.input["model_name"])
        # CRITICAL: custom tools answer with custom_tool_use_id, NOT tool_use_id.
        results.append({
            "type": "user.custom_tool_result",
            "custom_tool_use_id": use.id,
            "is_error": not outcome["ok"],
            "content": [{"type": "text", "text": outcome["content"]}],
        })
        print(f"attempt {attempts}: is_error={not outcome['ok']} -> {outcome['content']}")

    client.beta.sessions.events.send(session.id, events=results, betas=[BETA])

print(f"\nTotal tool attempts: {attempts} (expected 2: one transient error, one success).")

attempt 1: is_error=True -> {"error": "price source temporarily unavailable", "errorCategory": "transient", "isRetryable": true}


agent: BetaManagedAgentsAgentMessageEvent(id='sevt_01MMnaEq2svSMqeJcfbMnGFx', content=[BetaManagedAgentsTextBlock(text='Let me retry that:', type='text')], processed_at=datetime.datetime(2026, 7, 8, 16, 8, 27, 235023, tzinfo=TzInfo(0)), type='agent.message')
attempt 2: is_error=False -> {"model_name": "1959 Gibson Les Paul Standard", "price_usd": 275000}


agent: BetaManagedAgentsAgentMessageEvent(id='sevt_01XjUub1Mg8uzpXNAYwguRbJ', content=[BetaManagedAgentsTextBlock(text="A **1959 Gibson Les Paul Standard** is currently worth **$275,000 USD** on the used market.\n\nThis is one of the most coveted vintage guitars ever made, known for its exceptional tone, sustain, and collector value. The '59 Les Paul is considered the holy grail of electric guitars.", type='text')], processed_at=datetime.datetime(2026, 7, 8, 16, 8, 29, 223855, tzinfo=TzInfo(0)), type='agent.message')

Total tool attempts: 2 (expected 2: one transient error, one success).


### 5. Tear down (guarded)

Archive the session and the agent so nothing bills after the demo. There is **no `agents.delete`**; `archive` is the teardown verb.

**This cell is guarded by `RUN_TEARDOWN`** (set in the client cell above). It defaults to **OFF** so a "Run All" during class leaves the agent and session **live** for Console inspection. To archive, flip the switch and re-run this one cell, or set `ARCHIVE_ON_RUN=1` before launching (how the smoke tests force cleanup).

In [5]:
if RUN_TEARDOWN:
    client.beta.sessions.archive(session.id, betas=[BETA])
    client.beta.agents.archive(agent.id, betas=[BETA])
    print("Archived session and agent. Nothing left billing.")
else:
    print("Teardown SKIPPED - agent and session are still LIVE.")
    print("Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.")
    print("  agent:  ", agent.id)
    print("  session:", session.id)

Teardown SKIPPED - agent and session are still LIVE.
Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.
  agent:   agent_01XykSuqhFL3NkD25oJR9yqU
  session: sesn_0182ZQY7sVv5qgSvrAssToCM
